# WACV-workshop-skincon _ Oct 18
# All 8 Architechture

# Verify the Kaggle input files

In [ ]:
import os

for root, dirs, files in os.walk(os.environ.get('DATA_ROOT', './data')):
    for file in files:
        print(os.path.join(root, file))

# Verify the prediction files and split

In [ ]:
# Verify the prediction files and split

import numpy as np
import pandas as pd
import os

ARCH = "MViTv2"  # change when switching architecture
# ARCH = "ResNet50"
# ARCH = "ResNet101"
# ARCH = "ConvNeXt-tiny"
# ARCH = "DeiT-small"
# ARCH = "SwinV2-tiny"
# ARCH = "DINOv2_ViT"
# ARCH = "MaxViT-tiny"

base = os.environ.get("PREDICTION_DIR", "./data/val-mvitv2-files/")  # Configure for your environment








base = os.path.join(base, "")
probs = np.load(base + "val_probs.npy")
yhard = np.load(base + "val_yhard.npy")
ysoft = np.load(base + "val_ysoft.npy")
ids = np.load(base + "val_image_ids.npy", allow_pickle=True)
split = pd.read_csv(base + "skincon_calibration_evaluation_split_seed2027.csv")

print("Probabilities:", probs.shape)
print("Hard labels:", yhard.shape)
print("Soft labels:", ysoft.shape)
print("Image IDs:", ids.shape)
print("Split columns:", split.columns.tolist())
print(split.head())

# Confirm all predictions and labels refer to the same images
assert len(probs) == len(yhard) == len(ysoft) == len(ids), "File lengths do not match"
assert probs.shape[1] == ysoft.shape[1], "Class count mismatch"


# Step 3: Check image IDs and calibration/evaluation split

In [ ]:
# Step 3: Check image IDs and calibration/evaluation split

ids_clean = [str(x).strip() for x in ids]
split_ids = split["image_id"].astype(str).str.strip().tolist()

print("Duplicate prediction IDs:", len(ids_clean) - len(set(ids_clean)))
print("Duplicate split IDs:", len(split_ids) - len(set(split_ids)))
print("IDs missing from split:", len(set(ids_clean) - set(split_ids)))
print("Extra IDs in split:", len(set(split_ids) - set(ids_clean)))

print("\nSplit counts:")
print(split["split"].value_counts())

cal_ids = set(split.loc[split["split"] == "calibration", "image_id"])
eval_ids = set(split.loc[split["split"] == "evaluation", "image_id"])

print("\nCalibration/evaluation overlap:", len(cal_ids & eval_ids))

label_map = dict(zip(ids_clean, yhard))
label_mismatches = sum(
    int(label_map[image_id]) != int(label)
    for image_id, label in zip(split["image_id"], split["hard_label_index"])
    if image_id in label_map
)
print("Hard-label mismatches:", label_mismatches)


In [ ]:

# Step 4: Separate calibration and evaluation data

id_to_index = {str(image_id).strip(): i for i, image_id in enumerate(ids)}

cal_idx = [id_to_index[x] for x in split.loc[split["split"] == "calibration", "image_id"]]
eval_idx = [id_to_index[x] for x in split.loc[split["split"] == "evaluation", "image_id"]]

cal_probs, eval_probs = probs[cal_idx], probs[eval_idx]
cal_yhard, eval_yhard = yhard[cal_idx], yhard[eval_idx]
cal_ysoft, eval_ysoft = ysoft[cal_idx], ysoft[eval_idx]

print("Calibration probabilities:", cal_probs.shape)
print("Evaluation probabilities:", eval_probs.shape)
print("Calibration labels:", cal_yhard.shape)
print("Evaluation labels:", eval_yhard.shape)


In [ ]:
# Step 5: Validate prediction probabilities and labels

import numpy as np

print("All probabilities finite:", np.isfinite(probs).all())
print("Minimum probability:", probs.min())
print("Maximum probability:", probs.max())
print("Probability sums close to 1:",
      np.allclose(probs.sum(axis=1), 1, atol=1e-4))

print("Hard-label classes:", np.unique(yhard))
print("Labels within 0–7:",
      np.all((yhard >= 0) & (yhard < 8)))

print("All soft labels finite:", np.isfinite(ysoft).all())
print("Soft-label sums close to 1:",
      np.allclose(ysoft.sum(axis=1), 1, atol=1e-4))


In [ ]:
# Step 6: Calibrate Baseline prediction sets (alpha = 0.10)

alpha = 0.10
cal_true_probs = cal_probs[np.arange(len(cal_yhard)), cal_yhard.astype(int)]
cal_scores = 1.0 - cal_true_probs
n_cal = len(cal_scores)
k = int(np.ceil((n_cal + 1) * (1 - alpha)))
baseline_threshold = np.sort(cal_scores)[k - 1] if k <= n_cal else np.inf

prediction_sets = (1.0 - eval_probs) <= baseline_threshold
print("Baseline threshold:", round(float(baseline_threshold), 6))
print("Prediction sets:", prediction_sets.shape)


In [ ]:
# Step 7: Complete Baseline Evaluation

import pandas as pd

set_sizes = prediction_sets.sum(axis=1)
covered = prediction_sets[
    np.arange(len(eval_yhard)), eval_yhard.astype(int)
]

coverage = covered.mean()
avg_set_size = set_sizes.mean()
empty_rate = (set_sizes == 0).mean()
referral_rate = (set_sizes != 1).mean()

accepted = (set_sizes == 1)
accepted_accuracy = covered[accepted].mean() if accepted.any() else np.nan

confidence = eval_probs.max(axis=1)
predicted = eval_probs.argmax(axis=1)
correct = (predicted == eval_yhard)

ece = 0.0
for i in range(10):
    lower, upper = i / 10, (i + 1) / 10
    mask = (confidence >= lower) & (
        confidence < upper if i < 9 else confidence <= upper
    )
    if mask.any():
        ece += mask.mean() * abs(
            correct[mask].mean() - confidence[mask].mean()
        )

one_hot = np.eye(eval_probs.shape[1])[eval_yhard.astype(int)]
brier = np.mean(np.sum((eval_probs - one_hot) ** 2, axis=1))

eps = 1e-12
p = np.clip(eval_ysoft, eps, 1)
q = np.clip(eval_probs, eps, 1)
kl_values = np.sum(p * np.log(p / q), axis=1)

human_top = eval_ysoft.argmax(axis=1)
human_hit = prediction_sets[
    np.arange(len(human_top)), human_top
].mean()

results = pd.DataFrame({
    "Method": ["Baseline"] * 9,
    "Metric": [
        "Coverage",
        "Average Set Size",
        "Empty Set Rate",
        "Referral Rate",
        "Accepted-Case Accuracy",
        "ECE",
        "Brier Score",
        "Mean KL Divergence",
        "Human Top-Label Hit Rate"
    ],
    "Value": [
        coverage,
        avg_set_size,
        empty_rate,
        referral_rate,
        accepted_accuracy,
        ece,
        brier,
        kl_values.mean(),
        human_hit
    ]
})

baseline_results = results.copy()  # Save Baseline metrics for Step 12
print(baseline_results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
# Step 8: RAPS Evaluation

alpha = 0.10
lambda_reg = 0.01
k_reg = 5

def raps_scores(probabilities):
    order = np.argsort(-probabilities, axis=1)
    sorted_probs = np.take_along_axis(probabilities, order, axis=1)

    cumulative = np.cumsum(sorted_probs, axis=1)
    ranks = np.arange(1, probabilities.shape[1] + 1)
    penalty = lambda_reg * np.maximum(ranks - k_reg, 0)

    sorted_scores = cumulative + penalty

    scores = np.empty_like(sorted_scores)
    np.put_along_axis(scores, order, sorted_scores, axis=1)

    return scores

cal_raps_scores = raps_scores(cal_probs)
true_scores = cal_raps_scores[
    np.arange(len(cal_yhard)), cal_yhard.astype(int)
]

n = len(true_scores)
k = int(np.ceil((n + 1) * (1 - alpha)))
raps_threshold = np.sort(true_scores)[k - 1] if k <= n else np.inf

eval_raps_scores = raps_scores(eval_probs)
raps_sets = eval_raps_scores <= raps_threshold

set_sizes = raps_sets.sum(axis=1)
covered = raps_sets[
    np.arange(len(eval_yhard)), eval_yhard.astype(int)
]

coverage = covered.mean()
avg_set_size = set_sizes.mean()
empty_rate = (set_sizes == 0).mean()
referral_rate = (set_sizes != 1).mean()

accepted = (set_sizes == 1)
accepted_accuracy = covered[accepted].mean() if accepted.any() else np.nan

confidence = eval_probs.max(axis=1)
predicted = eval_probs.argmax(axis=1)
correct = predicted == eval_yhard

ece = 0.0
for i in range(10):
    lower, upper = i / 10, (i + 1) / 10
    mask = (confidence >= lower) & (
        confidence < upper if i < 9 else confidence <= upper
    )
    if mask.any():
        ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())

one_hot = np.eye(eval_probs.shape[1])[eval_yhard.astype(int)]
brier = np.mean(np.sum((eval_probs - one_hot) ** 2, axis=1))

eps = 1e-12
p = np.clip(eval_ysoft, eps, 1)
q = np.clip(eval_probs, eps, 1)
kl_values = np.sum(p * np.log(p / q), axis=1)

human_top = eval_ysoft.argmax(axis=1)
human_hit = raps_sets[
    np.arange(len(human_top)), human_top
].mean()

raps_results = pd.DataFrame({
    "Method": ["RAPS"] * 9,
    "Metric": [
        "Coverage",
        "Average Set Size",
        "Empty Set Rate",
        "Referral Rate",
        "Accepted-Case Accuracy",
        "ECE",
        "Brier Score",
        "Mean KL Divergence",
        "Human Top-Label Hit Rate"
    ],
    "Value": [
        coverage,
        avg_set_size,
        empty_rate,
        referral_rate,
        accepted_accuracy,
        ece,
        brier,
        kl_values.mean(),
        human_hit
    ]
})

print("RAPS threshold:", round(float(raps_threshold), 6))
print(raps_results.to_string(
    index=False,
    float_format=lambda x: f"{x:.4f}"
))


In [ ]:
# Step 9: DRAPS evaluation

alpha = 0.10
cal_true_probs = cal_probs[
    np.arange(len(cal_yhard)),
    cal_yhard.astype(int)
]

n_cal = len(cal_true_probs)
rank = int(np.floor(alpha * n_cal))

draps_threshold = np.sort(cal_true_probs)[rank - 1]

draps_sets = eval_probs >= draps_threshold

n_eval = len(eval_yhard)
set_sizes = draps_sets.sum(axis=1)

covered = draps_sets[
    np.arange(n_eval),
    eval_yhard.astype(int)
]

coverage = covered.mean()
avg_set_size = set_sizes.mean()
empty_rate = (set_sizes == 0).mean()

accepted = set_sizes == 1
referral_rate = (~accepted).mean()

accepted_accuracy = (
    covered[accepted].mean()
    if accepted.any() else np.nan
)

confidence = eval_probs.max(axis=1)
predicted = eval_probs.argmax(axis=1)
correct = predicted == eval_yhard

ece = 0.0

for i in range(10):
    lower = i / 10
    upper = (i + 1) / 10

    mask = (confidence >= lower) & (
        confidence < upper if i < 9 else confidence <= upper
    )

    if mask.any():
        ece += mask.mean() * abs(
            correct[mask].mean() - confidence[mask].mean()
        )

one_hot = np.eye(eval_probs.shape[1])[
    eval_yhard.astype(int)
]

brier = np.mean(
    np.sum((eval_probs - one_hot) ** 2, axis=1)
)

eps = 1e-12

p = np.clip(eval_ysoft, eps, 1)
q = np.clip(eval_probs, eps, 1)

kl_values = np.sum(
    p * np.log(p / q),
    axis=1
)

human_top = eval_ysoft.argmax(axis=1)

human_hit = draps_sets[
    np.arange(n_eval),
    human_top
].mean()

draps_results = pd.DataFrame({
    "Method": ["DRAPS"] * 9,
    "Metric": [
        "Coverage",
        "Average Set Size",
        "Empty Set Rate",
        "Referral Rate",
        "Accepted-Case Accuracy",
        "ECE",
        "Brier Score",
        "Mean KL Divergence",
        "Human Top-Label Hit Rate"
    ],
    "Value": [
        coverage,
        avg_set_size,
        empty_rate,
        referral_rate,
        accepted_accuracy,
        ece,
        brier,
        kl_values.mean(),
        human_hit
    ]
})

print("DRAPS threshold:", round(float(draps_threshold), 6))

print(draps_results.to_string(
    index=False,
    float_format=lambda x: f"{x:.4f}"
))


In [ ]:
# Step 10: Original SkinCON Hit Rate

def skincon_hit_rate(sets, soft_labels):
    human_ranking = np.argsort(-soft_labels, axis=1)

    hit_rates = []

    for i in range(len(sets)):
        k = int(sets[i].sum())

        if k == 0:
            hit_rates.append(np.nan)
            continue

        top_k = human_ranking[i, :k]

        hit = sets[i, top_k].sum() / k
        hit_rates.append(hit)

    return np.nanmean(hit_rates), np.sum(np.isnan(hit_rates))

results = []

for method, sets in [
    ("Baseline", prediction_sets),
    ("RAPS", raps_sets),
    ("DRAPS", draps_sets)
]:
    hr, empty = skincon_hit_rate(sets, eval_ysoft)

    results.append({
        "Method": method,
        "Original Hit Rate": hr,
        "Excluded Empty Sets": empty
    })

hit_rate_results = pd.DataFrame(results)

print(hit_rate_results.to_string(
    index=False,
    float_format=lambda x: f"{x:.4f}"
))


In [ ]:
# Step 11: Soft-label-weighted variant from the original notebook

alpha = 0.10
lam = 0.01
k_reg = 5

def soft_variant_scores(probabilities):
    order = np.argsort(-probabilities, axis=1)
    sorted_p = np.take_along_axis(probabilities, order, axis=1)

    cumulative = np.cumsum(sorted_p, axis=1)
    ranks = np.arange(1, probabilities.shape[1] + 1)
    sorted_scores = (
        1.0 - cumulative
        + lam * np.maximum(ranks - k_reg, 0)
    )

    scores = np.empty_like(sorted_scores)
    np.put_along_axis(scores, order, sorted_scores, axis=1)
    return scores

cal_scores = soft_variant_scores(cal_probs)
weighted_scores = np.sum(cal_ysoft * cal_scores, axis=1)

n_cal = len(weighted_scores)
k = int(np.ceil((n_cal + 1) * (1 - alpha)))

soft_variant_threshold = (
    np.sort(weighted_scores)[k - 1]
    if k <= n_cal else np.inf
)

soft_variant_sets = np.zeros_like(eval_probs, dtype=bool)

for i, p in enumerate(eval_probs):
    order = np.argsort(-p)
    cumulative = 0.0

    for rank, cls in enumerate(order, start=1):
        cumulative += p[cls]
        soft_variant_sets[i, cls] = True

        score = (
            1.0 - cumulative
            + lam * max(rank - k_reg, 0)
        )

        if score <= soft_variant_threshold:
            break

n_eval = len(eval_yhard)
sizes = soft_variant_sets.sum(axis=1)
hits = soft_variant_sets[
    np.arange(n_eval), eval_yhard.astype(int)
]

singleton = sizes == 1
human_top = eval_ysoft.argmax(axis=1)

human_order = np.argsort(-eval_ysoft, axis=1)
topk_hits = []

for i in range(n_eval):
    k = int(sizes[i])
    topk_hits.append(
        soft_variant_sets[i, human_order[i, :k]].sum() / k
        if k > 0 else np.nan
    )

soft_variant_results = pd.DataFrame({
    "Method": ["Soft-label variant"] * 7,
    "Metric": [
        "Coverage",
        "Average Set Size",
        "Empty Set Rate",
        "Referral Rate",
        "Accepted-Case Accuracy",
        "Human Top-Label Hit Rate",
        "Original Hit Rate"
    ],
    "Value": [
        hits.mean(),
        sizes.mean(),
        (sizes == 0).mean(),
        (sizes != 1).mean(),
        hits[singleton].mean() if singleton.any() else np.nan,
        soft_variant_sets[np.arange(n_eval), human_top].mean(),
        np.nanmean(topk_hits)
    ]
})

print("Soft-label threshold:", round(float(soft_variant_threshold), 6))
print(soft_variant_results.to_string(
    index=False,
    float_format=lambda x: f"{x:.4f}"
))


In [ ]:
# Step 12: Save architecture evaluation results
import pandas as pd

all_results = pd.concat([
    baseline_results,
    raps_results,
    draps_results,
    soft_variant_results
], ignore_index=True)

all_results.to_csv(
    f"{ARCH}_evaluation_results.csv",
    index=False
)

print("Results saved successfully.")


In [ ]:

# Arrange metrics in the requested sequence
metric_order = [
    "ECE",
    "Brier Score",
    "Coverage",
    "Average Set Size",
    "Singleton Rate",
    "Original Hit Rate",
    "Soft Hit Rate",
    "Human Top-Label Hit Rate",
    "Mean KL Divergence",
    "Empty Set Rate",
    "Referral Rate",
    "Accepted-Case Accuracy",
    "Excluded Empty Sets"
]

combined = all_results.copy()
for method, sets in [("Baseline", prediction_sets), ("RAPS", raps_sets),
                     ("DRAPS", draps_sets), ("Soft-label variant", soft_variant_sets)]:
    hit, empty = skincon_hit_rate(sets, eval_ysoft)
    combined = combined.loc[~((combined["Method"] == method) &
                              (combined["Metric"] == "Original Hit Rate"))].copy()
    combined = pd.concat([combined, pd.DataFrame([
        {"Method": method, "Metric": "Original Hit Rate", "Value": hit},
        {"Method": method, "Metric": "Excluded Empty Sets", "Value": empty},
        {"Method": method, "Metric": "Singleton Rate", "Value": (sets.sum(axis=1) == 1).mean()},
    ])], ignore_index=True)

for metric in ["ECE", "Brier Score", "Mean KL Divergence"]:
    value = baseline_results.loc[baseline_results["Metric"] == metric, "Value"].iloc[0]
    combined = pd.concat([combined, pd.DataFrame([
        {"Method": "Soft-label variant", "Metric": metric, "Value": value}
    ])], ignore_index=True)

print(f"{ARCH} - SUMMARY")
summary = combined.pivot(
    index="Metric",
    columns="Method",
    values="Value"
)

remaining = [m for m in summary.index if m not in metric_order]
summary = summary.reindex(
    metric_order + remaining
).dropna(how="all")

summary = summary.reindex(columns=[
    "Baseline",
    "RAPS",
    "DRAPS",
    "Soft-label variant"
])

summary = summary.rename(index={
    "Average Set Size": "Avg Set Size",
    "Original Hit Rate": "Hit Rate"
})

path = f"{ARCH}_all_methods_all_metrics.csv"
summary.to_csv(path, float_format="%.6f")


print(summary.to_markdown(
    tablefmt="grid",
    floatfmt=".4f"
))

print("CSV saved:", path)
